# 集成学习第四课：AdaBoost

## 本课目标

理解 Boosting 的第一种经典算法 AdaBoost：它如何通过“错题加权”让后面的模型重点学习前面没学好的样本。

一句话版：

```text
AdaBoost = 多个弱模型串行学习 + 错样本权重变大 + 模型加权投票
```

## 1. 从 Bagging 到 Boosting

前面学习的 Bagging 是并行思想：

```text
多个模型彼此独立训练，最后投票 / 平均。
```

Boosting 是串行思想：

```text
第 1 个模型先学。
第 2 个模型重点学第 1 个模型错的地方。
第 3 个模型继续补前面没学好的地方。
...
最后把多个模型组合起来。
```

所以 Boosting 的核心不是“大家各学各的”，而是“一个接一个补短板”。

## 2. AdaBoost 的名字

AdaBoost 是 Adaptive Boosting 的缩写。

Adaptive 的意思是“自适应”。

它自适应在哪里？

```text
上一轮分错的样本，下一轮权重会变大。
上一轮分对的样本，下一轮权重会相对变小。
```

这样后面的弱学习器会更关注难样本。

## 3. 什么是弱学习器？

弱学习器是指能力不强，但比乱猜好一点的模型。

AdaBoost 里常用的弱学习器是很浅的决策树，尤其是树桩 decision stump。

树桩就是深度为 1 的决策树：

```text
只问一个问题，就给出分类结果。
```

例如：

```text
如果 x1 <= 2.5，预测 A 类；否则预测 B 类。
```

单个树桩很弱，但很多树桩按顺序补错，就能组合成更强的模型。

## 4. AdaBoost 的核心流程

AdaBoost 可以粗略理解成下面 5 步：

1. 初始化所有样本权重相同。
2. 训练一个弱学习器。
3. 看它分错了哪些样本。
4. 提高分错样本的权重，降低分对样本的相对权重。
5. 重复多轮，最后让所有弱学习器加权投票。

直觉版：

```text
先做一遍题
-> 错题变重要
-> 下一轮重点做错题
-> 再错的题继续变重要
-> 最后综合每一轮老师的判断
```

## 5. 样本权重演示

下面用一个小例子看样本权重如何变化。

假设有 6 个样本，第一轮模型分错了第 2 个和第 5 个样本。

我们先不追公式，只看直觉：错的样本权重增加，对的样本权重减少，然后重新归一化。

In [ ]:
import numpy as np

sample_weights = np.ones(6) / 6
wrong = np.array([False, True, False, False, True, False])

new_weights = sample_weights.copy()
new_weights[wrong] *= 2.0
new_weights[~wrong] *= 0.7
new_weights = new_weights / new_weights.sum()

sample_weights, wrong, new_weights

可以看到，被分错的样本权重变大了。

下一轮训练时，模型会更在意这些样本，因为它们在损失中占的比例更高。

## 6. 弱学习器也有权重

AdaBoost 不只是给样本加权，也会给每个弱学习器一个权重。

直觉是：

```text
表现好的弱学习器，最终投票声音更大。
表现差的弱学习器，最终投票声音更小。
```

如果某个弱学习器错误率很低，它就更可信；如果错误率接近乱猜，它就不该有太大发言权。

In [ ]:
# AdaBoost 中二分类弱学习器权重的简化计算
# error 越小，alpha 越大；error 越接近 0.5，alpha 越接近 0
errors = np.array([0.1, 0.2, 0.3, 0.45])
alphas = 0.5 * np.log((1 - errors) / errors)

list(zip(errors, alphas))

上面的 `alpha` 可以先理解成“投票权重”。

错误率越低，投票权重越大。

## 7. 加权投票

Bagging 和随机森林通常是多数投票。

AdaBoost 更像是加权投票：

```text
强一点的弱学习器，投票权重大。
弱一点的弱学习器，投票权重小。
```

二分类里常把类别写成 `-1` 和 `1`，最终看加权结果的正负号。

In [ ]:
# 三个弱学习器对同一个样本的预测：-1 或 1
weak_predictions = np.array([1, -1, 1])
model_weights = np.array([0.8, 0.3, 0.6])

weighted_sum = np.sum(weak_predictions * model_weights)
final_prediction = 1 if weighted_sum > 0 else -1

weighted_sum, final_prediction

## 8. sklearn 实战：AdaBoost 分类

下面用 `make_moons` 数据集比较：

- 单个树桩
- AdaBoost + 多个树桩
- 随机森林

这里随机森林只是作为参照，不是说它一定总赢或总输。

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score

X, y = make_moons(n_samples=600, noise=0.3, random_state=22)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=22, stratify=y
)

stump = DecisionTreeClassifier(max_depth=1, random_state=22)
stump.fit(X_train, y_train)

try:
    adaboost = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=22),
        n_estimators=100,
        learning_rate=0.5,
        random_state=22,
    )
except TypeError:
    adaboost = AdaBoostClassifier(
        base_estimator=DecisionTreeClassifier(max_depth=1, random_state=22),
        n_estimators=100,
        learning_rate=0.5,
        random_state=22,
    )

adaboost.fit(X_train, y_train)

forest = RandomForestClassifier(
    n_estimators=100,
    random_state=22,
    n_jobs=-1,
)
forest.fit(X_train, y_train)

scores = {
    "单个树桩": accuracy_score(y_test, stump.predict(X_test)),
    "AdaBoost": accuracy_score(y_test, adaboost.predict(X_test)),
    "随机森林": accuracy_score(y_test, forest.predict(X_test)),
}

scores

通常可以看到，单个树桩能力很弱，但多个树桩通过 AdaBoost 串行组合后，效果会明显提升。

## 9. 查看每轮弱学习器权重

训练好的 AdaBoost 模型里，可以查看每个弱学习器的权重：

```python
model.estimator_weights_
```

也可以查看每轮弱学习器的错误率：

```python
model.estimator_errors_
```

In [ ]:
first_10_weights = adaboost.estimator_weights_[:10]
first_10_errors = adaboost.estimator_errors_[:10]

list(zip(first_10_errors, first_10_weights))

错误率越低的弱学习器，通常权重越大。

这和前面讲的加权投票直觉是一致的。

## 10. `n_estimators` 和 `learning_rate`

AdaBoost 里最重要的两个参数：

| 参数 | 含义 |
|---|---|
| `n_estimators` | 弱学习器数量，也就是 Boosting 轮数 |
| `learning_rate` | 每个弱学习器贡献的缩放系数 |

直觉上：

- `n_estimators` 越大，模型越复杂。
- `learning_rate` 越小，每一步走得越谨慎。
- 小 `learning_rate` 通常需要配合更多 `n_estimators`。

这和梯度下降里的学习率有一点类似：步子太大可能不稳，步子小则需要更多轮。

In [ ]:
results = []

for n_estimators in [10, 50, 100, 200]:
    for learning_rate in [0.1, 0.5, 1.0]:
        try:
            model = AdaBoostClassifier(
                estimator=DecisionTreeClassifier(max_depth=1, random_state=22),
                n_estimators=n_estimators,
                learning_rate=learning_rate,
                random_state=22,
            )
        except TypeError:
            model = AdaBoostClassifier(
                base_estimator=DecisionTreeClassifier(max_depth=1, random_state=22),
                n_estimators=n_estimators,
                learning_rate=learning_rate,
                random_state=22,
            )

        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))
        results.append((n_estimators, learning_rate, acc))

results

观察结果时，不要机械记某个参数一定最好。

更重要的是建立感觉：

```text
轮数控制复杂度，学习率控制每轮贡献。
```

## 11. AdaBoost 的优缺点

优点：

- 思想清晰，适合理解 Boosting。
- 可以把很多弱学习器组合成强模型。
- 参数不算多。
- 不容易像单棵深树那样随意过拟合。

缺点：

- 对噪声和异常点比较敏感。
- 如果某些样本本身标注错了，它们会被反复加权。
- 实际工程中，GBDT、XGBoost、LightGBM 往往更常用。

为什么对噪声敏感？

```text
AdaBoost 会重点关注分错的样本。
如果这个样本本身就是错标或异常点，模型可能会被它牵着走。
```

## 12. Bagging、随机森林、AdaBoost 对比

| 方法 | 训练方式 | 重点解决 | 核心直觉 |
|---|---|---|---|
| Bagging | 并行 | 降低方差 | 多个模型看不同数据后投票 |
| 随机森林 | 并行 | 降低方差 | 样本随机 + 特征随机 + 多树投票 |
| AdaBoost | 串行 | 降低偏差 | 后面的模型重点修正前面的错误 |

简单说：

```text
Bagging 像让很多人独立做题再投票。
AdaBoost 像一个人不断整理错题，一个接一个补短板。
```

## 13. 本课小结

- AdaBoost 是 Boosting 的经典入门算法。
- 它按顺序训练多个弱学习器。
- 前一轮分错的样本，后一轮权重会变大。
- 表现好的弱学习器，在最终投票中权重更大。
- `n_estimators` 控制弱学习器数量。
- `learning_rate` 控制每个弱学习器的贡献大小。
- AdaBoost 对噪声和异常点比较敏感。

记忆方式：

```text
AdaBoost = 错题变重要 + 弱模型接力 + 加权投票。
```